In [6]:
"""
Telegram NASDAQ Scraper v3 — Canaux vérifiés
=============================================
Tous les usernames ont été vérifiés sur t.me
pip install telethon pandas tqdm nest_asyncio
"""

import nest_asyncio
nest_asyncio.apply()

import asyncio
import pandas as pd
import re
from datetime import datetime, timezone
from tqdm import tqdm
from telethon import TelegramClient

# ============================================================
# CONFIGURATION
# ============================================================

API_ID   = 34806017
API_HASH = "d8973e5b317e0808635e54d81e543379"
PHONE    = "+33750013431"

# ✅ Canaux VÉRIFIÉS sur t.me (usernames exacts)
CHANNELS = [
    # --- Canaux NASDAQ/NAS100 directs ---
    "nas100masters",                      # NasdaqMasters — signaux NAS100 (178k membres)
    "nas100_trading_signal",              # NAS100 free signals — très actif
    "nasdaq_freesignals",                 # NASDAQ TRADERS — signaux quotidiens
    "EagleSpiritforexacademyNASDAQ100",   # Eagle Spirit NASDAQ100
    "FREEUS30XAUUSDSIGNALS",              # US30 + NAS100 signals
    "Nas100_Trading_forex_signal",        # NAS100 forex signals

    # --- Canaux finance générale très actifs ---
    "financialjuice",          # Financial Juice — news marché 24/7 (NASDAQ mentionné constamment)
    "TradingViewIdeas",        # TradingView Ideas — déjà 105 msgs trouvés ✅

    # --- Finance / marchés US ---
    "investing_com",           # Investing.com officiel
    "forexsignalsfactory",     # Forex + indices signals
    "wallstreetmemes",         # Humour + opinion marché
    "marketsignals",           # Market signals

    "vanillafinancenews",
    "FREEDOMFINANCE",
    "money",
    "finance",
    "curvefi",
    "personalfinancesg",
    "finance1",
    "SerezhaCalls",
    "nasdaq_r",
    "stocks",
    "x_stocks_bot",
    "stocks_0"
]

START_DATE  = datetime(2026, 1, 1, tzinfo=timezone.utc)
END_DATE    = datetime(2026, 4, 1, 23, 59, 59, tzinfo=timezone.utc)
OUTPUT_FILE = "telegram_nasdaq_jan_mars_2026_new.csv"

NASDAQ_RE = re.compile(
    r'\b(nasdaq|qqq|ndx|nq|nas100|nasdaq100|tech stock|nvda|aapl|msft|'
    r'googl|amzn|meta|tsla|faang|mag7|magnificent seven|sp500|s&p 500|'
    r'dow jones|us30|ustec|buy|sell|bullish|bearish|long|short)\b',
    re.IGNORECASE
)

# ============================================================
# COLLECTE
# ============================================================

async def collect():
    client = TelegramClient("nasdaq_session", API_ID, API_HASH)
    await client.start(phone=PHONE)
    print("✅ Connecté à Telegram !\n")

    all_posts = []
    ok_channels = []
    fail_channels = []

    for channel_name in tqdm(CHANNELS, desc="Canaux"):
        try:
            entity = await client.get_entity(channel_name)
            title  = getattr(entity, "title", channel_name)
            print(f"\n📡 @{channel_name} — {title}")

            count = 0
            total_checked = 0

            async for msg in client.iter_messages(entity, offset_date=END_DATE, reverse=False):
                if msg.date < START_DATE:
                    break

                total_checked += 1
                if not msg.text:
                    continue

                # Filtre NASDAQ — élargi pour les canaux de signaux
                if not NASDAQ_RE.search(msg.text):
                    continue

                all_posts.append({
                    "id":       f"tg_{channel_name}_{msg.id}",
                    "channel":  channel_name,
                    "date":     msg.date.strftime("%Y-%m-%d"),
                    "datetime": msg.date.strftime("%Y-%m-%d %H:%M:%S"),
                    "text":     msg.text.replace("\n", " ")[:600],
                    "views":    getattr(msg, "views", 0) or 0,
                    "forwards": getattr(msg, "forwards", 0) or 0,
                    "url":      f"https://t.me/{channel_name}/{msg.id}",
                })
                count += 1

            print(f"   → {count} msgs NASDAQ / {total_checked} total")
            ok_channels.append(channel_name)

        except Exception as e:
            print(f"   ⚠️  @{channel_name} : {e}")
            fail_channels.append(channel_name)

    await client.disconnect()

    print(f"\n✅ Canaux OK    : {ok_channels}")
    print(f"❌ Canaux KO    : {fail_channels}")
    return all_posts

# ============================================================
# LANCEMENT
# ============================================================

posts = asyncio.run(collect())

if posts:
    df = pd.DataFrame(posts).drop_duplicates("id").sort_values("datetime").reset_index(drop=True)
    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    daily = df.groupby("date").size()
    print(f"\n✅ {len(df)} messages → {OUTPUT_FILE}")
    print(f"   Jours couverts : {df['date'].nunique()} / 90")
    print(f"   Jours ≥ 10     : {(daily >= 10).sum()}")
    print(f"   Top canaux     : {df['channel'].value_counts().head(5).to_dict()}")
    print(df[["date", "channel", "text"]].head(3).to_string(index=False))
else:
    print("❌ Aucun message.")

✅ Connecté à Telegram !



Canaux:   0%|          | 0/24 [00:00<?, ?it/s]

   ⚠️  @nas100masters : No user has "nas100masters" as username

📡 @nas100_trading_signal — Nasdaq 100 index


Canaux:   8%|▊         | 2/24 [00:00<00:04,  4.82it/s]

   → 0 msgs NASDAQ / 0 total

📡 @nasdaq_freesignals — NASDAQ TRADERS


...

   → 315 msgs NASDAQ / 1420 total

📡 @stocks — Stocks - Live Feeds ▶️


Canaux:  96%|█████████▌| 23/24 [03:56<00:07,  7.57s/it]

   → 131 msgs NASDAQ / 217 total

📡 @x_stocks_bot — x_stocks_bot
   → 0 msgs NASDAQ / 0 total


Canaux: 100%|██████████| 24/24 [03:56<00:00,  9.86s/it]


   ⚠️  @stocks_0 : Nobody is using this username, or the username is unacceptable. If the latter, it must match r"[a-zA-Z][\w\d]{3,30}[a-zA-Z\d]" (caused by ResolveUsernameRequest)

✅ Canaux OK    : ['nas100_trading_signal', 'nasdaq_freesignals', 'EagleSpiritforexacademyNASDAQ100', 'FREEUS30XAUUSDSIGNALS', 'Nas100_Trading_forex_signal', 'financialjuice', 'TradingViewIdeas', 'forexsignalsfactory', 'wallstreetmemes', 'vanillafinancenews', 'FREEDOMFINANCE', 'money', 'finance', 'curvefi', 'personalfinancesg', 'finance1', 'SerezhaCalls', 'nasdaq_r', 'stocks', 'x_stocks_bot']
❌ Canaux KO    : ['nas100masters', 'investing_com', 'marketsignals', 'stocks_0']

✅ 3864 messages → telegram_nasdaq_jan_mars_2026_new.csv
   Jours couverts : 91 / 90
   Jours ≥ 10     : 91
   Top canaux     : {'TradingViewIdeas': 2372, 'financialjuice': 709, 'nasdaq_r': 315, 'stocks': 131, 'finance1': 128}
      date          channel                                                                                        